## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here


## B 长跑

In [ ]:
import sys
import heapq

INF = 10**18

def solve_one(n, L, Maxn, S, stations):
    if L <= Maxn:
        return "Yes"

    # 同一位置可能有多个补给站，只保留最便宜的
    mp = {}
    for p, c in stations:
        if 0 < p < L:
            if p not in mp or c < mp[p]:
                mp[p] = c

    points = [(0, 0)]
    for p in sorted(mp):
        points.append((p, mp[p]))
    points.append((L, 0))

    # 堆中存储：(当前最小花费, 位置)
    heap = [(0, 0)]

    for i in range(1, len(points)):
        pos, cost = points[i]

        # 移除跑不到当前位置的点
        while heap and heap[0][1] < pos - Maxn:
            heapq.heappop(heap)

        if not heap:
            return "No"

        cur_cost = heap[0][0] + cost

        # 如果当前位置是终点
        if pos == L:
            return "Yes" if cur_cost <= S else "No"

        # 超过预算的状态没必要继续扩展
        if cur_cost <= S:
            heapq.heappush(heap, (cur_cost, pos))

    return "No"


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    idx = 0
    ans = []

    while idx < len(data):
        n = data[idx]
        L = data[idx + 1]
        Maxn = data[idx + 2]
        S = data[idx + 3]
        idx += 4

        stations = []
        for _ in range(n):
            p = data[idx]
            c = data[idx + 1]
            idx += 2
            stations.append((p, c))

        ans.append(solve_one(n, L, Maxn, S, stations))

    print("\n".join(ans))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import sys

MASK = (1 << 64) - 1
BASE = 911382323


def manacher(s):
    n = len(s)

    # d1[i]：以 i 为中心的奇回文半径
    d1 = [0] * n
    l, r = 0, -1
    for i in range(n):
        k = 1 if i > r else min(d1[l + r - i], r - i + 1)

        while i - k >= 0 and i + k < n and s[i - k] == s[i + k]:
            k += 1

        d1[i] = k

        if i + k - 1 > r:
            l = i - k + 1
            r = i + k - 1

    # d2[i]：以 i - 1 和 i 中间为中心的偶回文半径
    d2 = [0] * n
    l, r = 0, -1
    for i in range(n):
        k = 0 if i > r else min(d2[l + r - i + 1], r - i + 1)

        while i - k - 1 >= 0 and i + k < n and s[i - k - 1] == s[i + k]:
            k += 1

        d2[i] = k

        if i + k - 1 > r:
            l = i - k
            r = i + k - 1

    return d1, d2


def solve(n, A, B):
    d1A, d2A = manacher(A)
    d1B, d2B = manacher(B)

    ans = 1

    # 只取 A 中的回文串
    for x in d1A:
        v = 2 * x - 1
        if v > ans:
            ans = v

    for x in d2A:
        v = 2 * x
        if v > ans:
            ans = v

    # 只取 B 中的回文串
    for x in d1B:
        v = 2 * x - 1
        if v > ans:
            ans = v

    for x in d2B:
        v = 2 * x
        if v > ans:
            ans = v

    max_ans = n + 1
    if ans == max_ans:
        return ans

    RA = A[::-1]

    power = [1] * (n + 1)
    hRA = [0] * (n + 1)
    hB = [0] * (n + 1)

    for i in range(n):
        power[i + 1] = (power[i] * BASE) & MASK
        hRA[i + 1] = (hRA[i] * BASE + RA[i] + 1) & MASK
        hB[i + 1] = (hB[i] * BASE + B[i] + 1) & MASK

    def get_hash(h, l, length):
        return (h[l + length] - ((h[l] * power[length]) & MASK)) & MASK

    # 从 A[q] 向左，B[p] 向右，最多匹配多少个字符
    # q, p 都是 1-based
    def match(q, p):
        if q <= 0 or p > n:
            return 0

        limit = q
        right_len = n - p + 1
        if right_len < limit:
            limit = right_len

        if limit <= 0:
            return 0

        ra_start = n - q
        b_start = p - 1

        # 第一个字符不同，直接失败
        if RA[ra_start] != B[b_start]:
            return 0

        # 整段都相同，直接返回，避免二分
        if get_hash(hRA, ra_start, limit) == get_hash(hB, b_start, limit):
            return limit

        left, right = 1, limit - 1

        while left < right:
            mid = (left + right + 1) >> 1

            if get_hash(hRA, ra_start, mid) == get_hash(hB, b_start, mid):
                left = mid
            else:
                right = mid - 1

        return left

    def update(base_len, q, p):
        nonlocal ans

        if q <= 0 or p > n:
            return

        limit = q
        right_len = n - p + 1
        if right_len < limit:
            limit = right_len

        # 即使全部匹配，也不能超过当前答案，直接跳过
        if base_len + 2 * limit <= ans:
            return

        t = match(q, p)
        val = base_len + 2 * t

        if val > ans:
            ans = val

    # 情况 1：中间为空
    for k in range(1, n + 1):
        update(0, k, k)
        if ans == max_ans:
            return ans

    # 情况 2：中间回文串在 A 中，奇数长度
    for i, r in enumerate(d1A):
        c = i + 1
        base_len = 2 * r - 1

        q = c - r
        p = c + r - 1

        update(base_len, q, p)

        if ans == max_ans:
            return ans

    # 情况 3：中间回文串在 A 中，偶数长度
    for i, r in enumerate(d2A):
        if r == 0:
            continue

        base_len = 2 * r

        q = i - r
        p = i + r

        update(base_len, q, p)

        if ans == max_ans:
            return ans

    # 情况 4：中间回文串在 B 中，奇数长度
    for i, r in enumerate(d1B):
        c = i + 1
        base_len = 2 * r - 1

        q = c - r + 1
        p = c + r

        update(base_len, q, p)

        if ans == max_ans:
            return ans

    # 情况 5：中间回文串在 B 中，偶数长度
    for i, r in enumerate(d2B):
        if r == 0:
            continue

        base_len = 2 * r

        q = i - r + 1
        p = i + r + 1

        update(base_len, q, p)

        if ans == max_ans:
            return ans

    return ans


def main():
    data = sys.stdin.buffer.read().split()

    n = int(data[0])
    A = data[1]
    B = data[2]

    print(solve(n, A, B))


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
import sys


MAX_X = 100000


def solve_case(m, reader):
    bit = [0] * (m + 2)
    last = [0] * (MAX_X + 1)

    q_total = 0
    ans = -1
    top = 1 << ((m + 1).bit_length() - 1)

    def add(i, v):
        while i <= m:
            bit[i] += v
            i += i & -i

    def prefix_sum(i):
        s = 0
        while i > 0:
            s += bit[i]
            i -= i & -i
        return s

    def find_kth(k):
        idx = 0
        step = top
        while step:
            nxt = idx + step
            if nxt <= m and bit[nxt] < k:
                idx = nxt
                k -= bit[nxt]
            step >>= 1
        return idx + 1

    def take_question_from(pos):
        """
        找到位置 >= pos 的第一个还没用过的 ?
        找到就删除并返回 True，否则返回 False
        """
        nonlocal q_total

        if pos < 1:
            pos = 1

        before = prefix_sum(pos - 1)

        if q_total <= before:
            return False

        q_pos = find_kth(before + 1)
        add(q_pos, -1)
        q_total -= 1
        return True

    for i in range(1, m + 1):
        parts = reader.readline().split()

        if ans != -1:
            continue

        op = parts[0]

        # 兼容 ASCII '?' 和中文全角 '？'
        if op != b'I' and op != b'O':
            add(i, 1)
            q_total += 1
            continue

        x = int(parts[1])
        pre = last[x]

        if op == b'I':
            # 上一次也是 I x，中间必须补一个 O x
            if pre > 0:
                if not take_question_from(pre + 1):
                    ans = i
                    continue

            last[x] = i

        else:
            # 上一次不是 I x，则当前 O x 前面必须补一个 I x
            if pre <= 0:
                need_after = -pre + 1
                if not take_question_from(need_after):
                    ans = i
                    continue

            last[x] = -i

    return ans


def main():
    reader = sys.stdin.buffer
    outputs = []

    while True:
        line = reader.readline()
        if not line:
            break

        line = line.strip()
        if not line:
            continue

        m = int(line)

        if m == 0:
            outputs.append("-1")
            continue

        outputs.append(str(solve_case(m, reader)))

    print("\n".join(outputs))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
#include<bits/stdc++.h>
using namespace std;

int pre[105], x[105], y[105], n;

// 初始化并查集
void init(int n) {
    for(int i = 1; i <= n; i++) pre[i] = i;
}

// 查找根节点（带路径压缩）
int find(int x) {
    return pre[x] == x ? x : pre[x] = find(pre[x]);
}

// 合并两个集合
void merge(int a, int b) {
    int fa = find(a), fb = find(b);
    if(fa != fb) pre[fa] = fb;
}

int main() {
    ios::sync_with_stdio(0);
    cin.tie(0);
    
    cin >> n;
    init(n);
    
    // 读入坐标
    for(int i = 1; i <= n; i++) {
        cin >> x[i] >> y[i];
    }
    
    // 合并同行或同列的点
    for(int i = 1; i <= n; i++) {
        for(int j = 1; j <= n; j++) {
            if(x[i] == x[j] || y[i] == y[j]) {
                merge(i, j);
            }
        }
    }
    
    // 统计不同的连通块数
    int ans = 0;
    for(int i = 1; i <= n; i++) {
        if(pre[i] == i) ans++;
    }
    
    cout << ans - 1 << endl;
    return 0;
}

## F 通配符匹配

In [ ]:
import sys

Q = ord('?')


def build_segment(seg):
    """
    把一个不含 * 的段拆成若干字母块。
    例如 b"ab??cd?e" -> 长度 8, blocks = [(0,b"ab"), (4,b"cd"), (7,b"e")]
    """
    blocks = []
    i = 0
    m = len(seg)

    while i < m:
        if seg[i] == Q:
            i += 1
        else:
            j = i
            while j < m and seg[j] != Q:
                j += 1
            blocks.append((i, seg[i:j]))
            i = j

    return len(seg), blocks


def match_at(segment, s, pos):
    """
    判断 segment 能否匹配 s[pos : pos + len(segment)]
    """
    length, blocks = segment

    if pos < 0 or pos + length > len(s):
        return False

    for offset, block in blocks:
        if not s.startswith(block, pos + offset):
            return False

    return True


def find_segment(segment, s, start, end_start):
    """
    在 s 中寻找 segment 的最早匹配位置。
    起点范围是 [start, end_start]
    """
    length, blocks = segment

    if start > end_start:
        return -1

    # 空段直接匹配
    if length == 0:
        return start

    # 全是 ?，可以匹配任意长度为 length 的位置
    if not blocks:
        return start

    # 选择出现次数最少的字母块作为锚点，减少枚举次数
    best_idx = -1
    best_cnt = 10**18

    for idx, (offset, block) in enumerate(blocks):
        left = start + offset
        right = end_start + offset + len(block)

        cnt = s.count(block, left, right)

        if cnt == 0:
            return -1

        if cnt < best_cnt:
            best_cnt = cnt
            best_idx = idx

    anchor_offset, anchor = blocks[best_idx]

    search_left = start + anchor_offset
    search_right = end_start + anchor_offset + len(anchor)

    pos = s.find(anchor, search_left, search_right)

    while pos != -1:
        real_start = pos - anchor_offset
        ok = True

        for idx, (offset, block) in enumerate(blocks):
            if idx == best_idx:
                continue

            if not s.startswith(block, real_start + offset):
                ok = False
                break

        if ok:
            return real_start

        pos = s.find(anchor, pos + 1, search_right)

    return -1


def can_match(pattern_has_star, segments, filename):
    s = filename
    n = len(s)

    # 没有 *，必须完整等长匹配
    if not pattern_has_star:
        only = segments[0]
        if only[0] != n:
            return False
        return match_at(only, s, 0)

    prefix = segments[0]
    suffix = segments[-1]
    middle_segments = segments[1:-1]

    prefix_len = prefix[0]
    suffix_len = suffix[0]

    # 前缀和后缀不能重叠
    suffix_start = n - suffix_len

    if suffix_start < prefix_len:
        return False

    # 匹配前缀
    if not match_at(prefix, s, 0):
        return False

    # 匹配后缀
    if not match_at(suffix, s, suffix_start):
        return False

    cur = prefix_len
    limit = suffix_start

    # 中间段按顺序贪心找最早位置
    for seg in middle_segments:
        seg_len = seg[0]
        pos = find_segment(seg, s, cur, limit - seg_len)

        if pos == -1:
            return False

        cur = pos + seg_len

    return True


def main():
    data = sys.stdin.buffer.read().splitlines()

    pattern = data[0].strip()
    n = int(data[1])

    pattern_has_star = b'*' in pattern

    raw_segments = pattern.split(b'*')
    segments = [build_segment(seg) for seg in raw_segments]

    ans = []

    for i in range(n):
        filename = data[i + 2].strip()

        if can_match(pattern_has_star, segments, filename):
            ans.append("YES")
        else:
            ans.append("NO")

    print("\n".join(ans))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;

typedef long long ll;
const int N = 32;
int n;
ll f[N][3];  // f[i][x] 转移步数
int p[N][3];  // p[i][x] 转移目标柱子

int main() {
    cin >> n;
    
    // 读入优先级
    int vis[3] = {0};
    for (int i = 0; i < 6; i++) {
        string s;
        cin >> s;
        int from = s[0] - 'A';
        int to = s[1] - 'A';
        if (!vis[from]) {
            vis[from] = 1;
            p[1][from] = to; // 记录单个盘子的移法
        }
    }
    
    // 初始化 f[1]
    for (int i = 0; i < 3; i++) {
        f[1][i] = 1;
    }
    
    // DP 递推
    for (int i = 2; i <= n; i++) {
        for (int x = 0; x < 3; x++) {
            int y = p[i-1][x];
            int z = 3 - x - y;
            if (p[i-1][y] == z) {
                f[i][x] = f[i-1][x] + 1 + f[i-1][y];
                p[i][x] = z;
            } else if (p[i-1][y] == x) {
                f[i][x] = 2 * f[i-1][x] + 2 + f[i-1][y];
                p[i][x] = y;
            }
        }
    }
    
    // 答案：从 A（0）柱移动 n 个盘子的步数
    cout << f[n][0] << endl;
    
    return 0;
}

## H 马步距离

In [ ]:
#include <iostream>
#include <queue>
#include <algorithm>
using namespace std;
const int N = 150; // BFS地图大小
int dx[8] = {1,2,2,1,-1,-2,-2,-1};
int dy[8] = {2,1,-1,-2,-2,-1,1,2};
int dis[N][N]; // 存储步数

int bfs(int x, int y) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            dis[i][j] = -1;
        }
    }
    
    queue<pair<int, int>> q;
    dis[x][y] = 0;
    q.push({x, y});
    
    while (!q.empty()) {
        auto now = q.front(); q.pop();
        // 可以在这里提前判断是否到达目标点
        // if (now.first == 50 && now.second == 50) break;
        for (int i = 0; i < 8; i++) {
            int nx = now.first + dx[i];
            int ny = now.second + dy[i];
            if (nx >= 0 && nx < N && ny >= 0 && ny < N && dis[nx][ny] == -1) {
                dis[nx][ny] = dis[now.first][now.second] + 1;
                q.push({nx, ny});
            }
        }
    }
    
    return dis[50][50];
}

int main() {
    int xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;
    
    // 1. 转换坐标，确保为非负数
    int x = abs(xp - xs);
    int y = abs(yp - ys);
    int ans = 0;
    
    // 2. 大范围贪心
    while (x + y > 50) {
        if (x < y) swap(x, y);
        // 特殊贪心策略
        if (x - 4 >= 2 * y) {
            x -= 4;
            ans += 2;
        } else {
            x -= 2;
            y -= 1;
            ans++;
        }
        // 再次确保x >= y
        if (x < y) swap(x, y);
    }
    
    // 3. 坐标归一化并BFS
    x += 50;
    y += 50;
    int step = bfs(x, y);
    
    // 4. 输出结果
    cout << ans + step << endl;
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
#include <vector>
#include <stack>
#include <algorithm>
using namespace std;

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        // 技巧：在末尾加入一个高度为0的哨兵，确保栈中的元素都能被弹出计算
        heights.push_back(0);
        int n = heights.size();
        stack<int> st; // 存储元素索引，栈内索引对应的高度保持单调递增
        int maxArea = 0;
        
        for (int i = 0; i < n; i++) {
            // 当遇到一个比栈顶高度更小的柱子时，开始计算面积
            while (!st.empty() && heights[st.top()] > heights[i]) {
                int h = heights[st.top()];
                st.pop();
                // 关键：弹出栈顶后，新的栈顶就是左边第一个小于当前高度的柱子的索引
                int left = st.empty() ? -1 : st.top();
                // 当前遍历到的索引 i 就是右边第一个小于当前高度的柱子的索引
                int width = i - left - 1;
                maxArea = max(maxArea, h * width);
            }
            st.push(i);
        }
        return maxArea;
    }
};

## J 消防局的设立

In [ ]:
## add your code her#include <bits/stdc++.h>
using namespace std;

const int MAXN = 1005;
vector<int> g[MAXN];
int father[MAXN];
int depth[MAXN];
bool covered[MAXN];
int n;

void dfs(int u, int fa, int d) {
    father[u] = fa;
    depth[u] = d;
    for (int v : g[u]) {
        if (v == fa) continue;
        dfs(v, u, d+1);
    }
}

int main() {
    cin >> n;
    // 读入n-1行，第i行对应节点i+1? 注意输入描述: 第i行的正整数为a[i]，表示从编号为i的基地到编号为a[i]的基地之间有一条道路，有a[i] < i。所以i从1到n-1。
    for (int i = 1; i <= n-1; i++) {
        int a;
        cin >> a;
        g[i+1].push_back(a);
        g[a].push_back(i+1);
    }
    // 以1为根进行dfs
    dfs(1, 0, 0);
    // 按深度降序排序节点编号
    vector<int> nodes(n);
    iota(nodes.begin(), nodes.end(), 1);
    sort(nodes.begin(), nodes.end(), [&](int x, int y) {
        return depth[x] > depth[y];
    });
    int ans = 0;
    for (int u : nodes) {
        if (!covered[u]) {
            // 在爷爷节点放置消防站
            int fa = father[u];
            int grand = father[fa];
            if (grand == 0) grand = fa; // 如果没有爷爷，就在父亲放
            // 如果grand还是0（根节点的父亲不存在），就在根放
            if (grand == 0) grand = u;
            ans++;
            // 覆盖距离<=2的所有节点
            // 先标记grand和它的所有子节点（距离1），以及子节点的子节点（距离2）
            // 简单起见：BFS两层
            queue<int> q;
            q.push(grand);
            vector<int> dist(n+1, -1);
            dist[grand] = 0;
            while (!q.empty()) {
                int cur = q.front(); q.pop();
                if (dist[cur] > 2) continue;
                covered[cur] = true;
                for (int v : g[cur]) {
                    if (dist[v] == -1) {
                        dist[v] = dist[cur] + 1;
                        q.push(v);
                    }
                }
            }
        }
    }
    cout << ans << endl;
    return 0;
}e